# Overview
In this notebook, we will leverage custom APIs to extend OMOP tables that are shipped out of the box with the Healthcare Solutions offering. The custom API returns the mappings of the ids between the source and OMOP tables. These id mappings can then be used to extend OMOP tables. As a prerequisite, you must have run the relevant notebooks to ingest data into your bronze, silver, and OMOP lakehouses.

To illustrate how to use this API, we will extend the `provider` table in the OMOP lakehouse with a new column called `use_secure_message` that indicates whether or not the provider uses `Direct Secure Messaging`.

##### Configuration management and setup
To setup and manage configurations for Healthcare Solutions, please execute the following cell:

In [ ]:
%run %%msft_dm4h_setup_and_config_notebook%%

##### Install the required packages

In [ ]:
# HDS, DTT, TA4H, typing-extensions
%pip install azure-ai-textanalytics==5.3.0b2 opencensus-ext-azure==1.1.13 semantic-link==0.7.6 typing-extensions==4.8.0 tmp/hds_packages/dtt-0.2.0.591-py3-none-any.whl tmp/hds_packages/hds-0.3.2-py3-none-any.whl

In [ ]:
%run %%msft_dm4h_setup_and_config_notebook%% {"enable_spark_setup" : true, "enable_packages_mount" : false}

##### 1. Setup required values for mapping ids from source to the target table.
 

 - `SOURCE_SYSTEM_ID` (str): *Name of the source system.*
 - `SOURCE_TABLE` (str): *The source table name in the `silver_database` that contains the data we are using to extend the OMOP table.*
 - `TARGET_TABLE` (str): *The target OMOP table that we will extend.*
 - `SOURCE_ID_COLUMN` (str): *Column name of the id of the source table.*
 - `TARGET_ID_COLUMN` (str): *Column name for target table ids.*
 - `TRANSFORMED_EXTENSION_COLUMN_NAME` (str):  *Name of the column that is desired to be added to the omop target table.*
 - `SECONDARY_LAKE_LOCATION` (str): *Absolute location path of the secondary lake.*


In [ ]:
# Setup source and target table
SOURCE_SYSTEM_ID = "FHIR"
SOURCE_TABLE = "Practitioner"
TARGET_TABLE = "Provider"
SOURCE_ID_COLUMN = "id"
TARGET_ID_COLUMN = "provider_id"
TRANSFORMED_EXTENSION_COLUMN_NAME = "use_secure_message"
SECONDARY_LAKE_PATH = "DMHCheckpoint/dtt/dtt_state_db"
SECONDARY_LAKE_LOCATION = f"abfss://{workspace_name}@{one_lake_endpoint}/{solution_name}/{SECONDARY_LAKE_PATH}"

##### 2. Enable automerge so that delta write can evolve schema.

In [ ]:
from pyspark.sql import SparkSession

# Enable schema evolution
spark = (
    SparkSession.builder.appName("EnableAutoMerge")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    .getOrCreate()
)

##### 3. Setup source FHIR dataframe.

Our source dataframe will be derived from the `Practitioner` table and will consist of two columns:
1) `id`: *The id of the source table.*
2) `use_secure_message`: *This column will contain a boolean indicating whether the practitioner uses Direct Secure Messaging or not. The data in this column will be used to extend the*
*OMOP tables. Extensions, by default, are strings. We can, however, leverage the `parse_extension` utility to parse the extension on the `telecom` field in the `Practitioner`* *table in the silver lakehouse to retrieve the boolean value.*

In [ ]:
from microsoft.fabric.hls.hds.utils.extension_parser import ExtensionParser

# To enable fhir string extension
ExtensionParser.register(spark)

# Setup and view the Source dataframe
SOURCE_DF = spark.sql(
    f"""
    SELECT `{SOURCE_ID_COLUMN}`, cast(parse_extension(telecom[0].extension, 'http://hl7.org/fhir/us/core/StructureDefinition/us-core-direct', 'valueBoolean', '') as boolean) as `{TRANSFORMED_EXTENSION_COLUMN_NAME}` from
    `{silver_database_name}`.`{SOURCE_TABLE}` 
"""
)
print("------------ PRACTITIONER SOURCE DATAFRAME --------------")
display(SOURCE_DF.limit(10))

##### 4. Map the source system ids to ids of the ids_table_name table.

**Method**: `map_source_to_ids`

**Parameters**:
- `ids_table_name` (str): *Target table name.*
- `source_system_id` (str): *Name of the source system.*
- `source_df` (DataFrame): *A dataframe containing the id that needs to be mapped.*
- `source_internal_id_column_name` (str): *A name of the column containing the internal id of the source system.*
*Internal id is the id that identifies and entity in the source system and is unique within the source system.*
- `source_external_id_column_name` (str): *A name of the  containing the external id of the source system.*
*External id is the id that uniquely identifies an entity in the source system and is unique across all*
*source systems. This parameter is mandatory in case this API is used to synchronize data between multiple source systems.*
- `mapped_column_name` (str): *A name of the column in the output DataFrame that will contain the ids from the ids_table_name table that match the source system ids.*
- `secondary_lake_location` (str): *Absolute path of the location of the secondary lake.*

**Returns**:
        DataFrame: The original DataFrame with additional column containing the ids from the ids_table_name table that match the source system ids.



In [ ]:
from dmf.api.keys import map_source_to_ids
from pyspark.sql.functions import col

# Helper method to map FHIR <-> OMOP tables
# Usecase: Map practitioner_id to provider_id to transform secure email extension column from Practitioner to Provider table
PROVIDER_ID_MAPPED_DF = map_source_to_ids(
    ids_table_name=TARGET_TABLE,
    source_system_id=SOURCE_SYSTEM_ID,
    source_df=SOURCE_DF,
    source_internal_id_column_name=SOURCE_ID_COLUMN,
    source_external_id_column_name=None,
    mapped_column_name=TARGET_ID_COLUMN,
    secondary_lake_location=SECONDARY_LAKE_LOCATION,
)

# Display key mapping result dataframe
print("------------ KEY MAPPINGS RESULT DATAFRAME --------------")
PROVIDER_ID_MAPPED_DF = PROVIDER_ID_MAPPED_DF.select(
    col("id").alias("practitioner_id"),
    col("provider_id"),
    col(f"{TRANSFORMED_EXTENSION_COLUMN_NAME}"),
)
display(PROVIDER_ID_MAPPED_DF.limit(10))

##### 5. Extend OMOP table
Use the Delta merge API to extend the OMOP table with the new column.

In [ ]:
from delta.tables import DeltaTable

# Setup target delta table
TARGET_DELTA_TABLE = DeltaTable.forName(spark, f"{omop_database_name}.{TARGET_TABLE}")

TARGET_DELTA_TABLE.alias("target").merge(
    PROVIDER_ID_MAPPED_DF.alias("source"), "target.provider_id = source.provider_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

##### 6. Verify target column ID were added to the Dataframe.

In [ ]:
omop_provider_df = spark.sql(
    f"""
    SELECT `{TARGET_ID_COLUMN}`, `{TRANSFORMED_EXTENSION_COLUMN_NAME}`, * from
    `{omop_database_name}`.`{TARGET_TABLE}` 
"""
)

display(omop_provider_df.limit(10))